# Listen and Learn — Evaluation

**This notebook produces the figures for the dissertation.**


| Section | Question answered |
|---|---|
| 1 | Is it reproducible? |
| 2 | Are the categories any good? |
| 3 | Are the outputs correct? (precision, recall, F1) |
| 4 | Is it robust? |
| 5 | Summary table |
| 6 | *Optional:* how does a generative baseline compare? |

---

## The alignment problem

This system discovers its own categories. The SemEval-2014 laptop data has no
fixed taxonomy — it has 1,042 free-text **aspect terms** (`battery life`,
`cord`, `tech guy`) with character offsets.

They cannot be compared directly. The bridge is to map each gold term to its
nearest discovered category by embedding cosine similarity. That mapping is
deterministic and requires no human labelling.

The mapping contributes its own error,
so the figures are a *lower bound* on true performance.


**Runtime:** CPU is fine for sections 1–5. Section 6 needs a GPU.

## 0 · Setup

In [1]:
!pip install -q sentence-transformers==3.0.1 vaderSentiment
!python -m spacy download en_core_web_sm -q
print("Installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 125.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all t

In [2]:
import json, io, hashlib, time, random, re
import numpy as np
import pandas as pd
import torch, spacy
from sentence_transformers import SentenceTransformer
from google.colab import files

# Determinism locks — identical to the production pipeline
torch.use_deterministic_algorithms(True)
torch.set_grad_enabled(False)

DECIMALS = 6
print("Ready.")

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Ready.


In [3]:
print("Upload model artifact (mdl_*.json)")
up = files.upload()
artifact = json.loads(list(up.values())[0].decode("utf-8"))

print(f"\nModel      : {artifact['name']}")
print(f"Model ID   : {artifact['model_id']}")
print(f"Created    : {artifact['created_utc'][:10]}")
print(f"Categories : {len(artifact['categories'])}")
for c in artifact["categories"]:
    print(f"   {c['id']}  {c['name']}")

Upload model artifact (mdl_*.json)


Saving mdl_260825014042.json to mdl_260825014042.json

Model      : Laptop Review Analysis
Model ID   : mdl_260825014042
Created    : 2026-08-25
Categories : 10
   cat_001  Battery Life
   cat_002  Build Quality
   cat_003  Hardware Performance
   cat_004  User Interface Design
   cat_005  Ease Of Use
   cat_006  Upgrade Options
   cat_007  System Reliability
   cat_008  Device Size And Weight
   cat_009  Purchase And Value Perception
   cat_010  After Sales Support And Warranty


In [4]:
# Load the embedding model, verifying it matches the artifact.
# A mismatch means the vectors are not comparable and results would be wrong

EMBED_NAME = artifact["embedding"]["name"]
embedder = SentenceTransformer(EMBED_NAME, device="cpu")
embedder.eval()


def weights_fingerprint(model):
    h = hashlib.sha256()
    st = model.state_dict()
    for k in sorted(st.keys()):
        h.update(st[k].cpu().numpy().tobytes())
    return h.hexdigest()


expected = artifact["embedding"].get("weights_sha256")
if expected:
    actual = weights_fingerprint(embedder)
    if actual != expected:
        raise RuntimeError(
            "Embedding model does not match the one used to build this "
            "artifact. Vectors are not comparable."
        )
    print("Weights fingerprint verified.")


def embed(texts):
    """One at a time — no batch padding, no cross-contamination."""
    out = []
    for t in texts:
        v = embedder.encode([t], batch_size=1, convert_to_numpy=True,
                            normalize_embeddings=True,
                            show_progress_bar=False)[0]
        out.append(np.round(v, DECIMALS))
    return np.vstack(out)


print(f"Embedding model: {EMBED_NAME}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Weights fingerprint verified.
Embedding model: BAAI/bge-base-en-v1.5


In [5]:
# ── Segmentation and inference, identical to production ───────
# Dependency-based clause splitting: coordination is resolved by the
# parse rather than by matching connective words, so "and" splits
# "bright and the battery lasts" but not "bright and clear".

nlp = spacy.load("en_core_web_sm")

seg_cfg = artifact.get("segmentation", {})
HARD_BREAKS = set(seg_cfg.get("hard_breaks", [";", "\u2014", "\u2013"]))
LEADING_JOINERS = set(seg_cfg.get("leading_joiners",
    ["and", "but", "or", "yet", "so", "however",
     "although", "though", "whereas", "nor"]))
MIN_WORDS = seg_cfg.get("min_words", 3)


def _has_modifier(token):
    for child in token.children:
        if child.dep_ in ("amod", "advmod"):
            return True
        if child.dep_ == "compound":
            for gc in child.children:
                if gc.dep_ in ("amod", "advmod"):
                    return True
    return False


def _is_clausal_conjunct(token):
    if token.pos_ in ("VERB", "AUX"):
        return True
    for child in token.children:
        if child.dep_ in ("nsubj", "nsubjpass"):
            return True
    if token.pos_ in ("NOUN", "PROPN"):
        if _has_modifier(token) and _has_modifier(token.head):
            return True
    return False


def _strip_leading_joiner(text):
    text = text.lstrip(" ,;\u2014\u2013")
    parts = text.split(None, 1)
    if parts and parts[0].lower().strip(",") in LEADING_JOINERS:
        text = parts[1] if len(parts) > 1 else ""
    return text.strip(" ,;\u2014\u2013").strip()


def segment(comment):
    doc = nlp(str(comment).strip())
    clauses = []
    for sentence in doc.sents:
        breaks = set()
        for token in sentence:
            if token.dep_ == "conj" and _is_clausal_conjunct(token):
                start = min(t.i for t in token.subtree)
                if start > sentence.start:
                    prev = doc[start - 1]
                    if prev.dep_ == "cc" or prev.text in {",", ";"}:
                        start -= 1
                breaks.add(start)
            elif token.text in HARD_BREAKS:
                breaks.add(token.i + 1)
        cuts = sorted(breaks | {sentence.start, sentence.end})
        for a, b in zip(cuts, cuts[1:]):
            t = _strip_leading_joiner(doc[a:b].text)
            if t:
                clauses.append(t)
    out = [c for c in clauses if len(c.split()) >= MIN_WORDS]
    return out if out else [str(comment).strip()]


CENTRE  = np.array(artifact["centering_vector"])
CATS    = artifact["categories"]
CAT_IDS = [c["id"] for c in CATS]
ANCHORS = {c["id"]: np.array(c["anchor"]) for c in CATS}
AXES    = {c["id"]: np.array(c["sentiment_axis"]) for c in CATS}
THRESH  = {c["id"]: c["threshold"] for c in CATS}
SCALES  = {c["id"]: c["sentiment_scale"] for c in CATS}
NAMES   = {c["id"]: c["name"] for c in CATS}


def rescale(raw, scale):
    p5, p95 = scale["p5"], scale["p95"]
    if p95 == p5:
        return 0.0
    return round(float(np.clip((raw - p5) / (p95 - p5) * 2 - 1, -1, 1)), 4)


def analyse(comment):
    """Analyse ONE comment in isolation. No dataset, no neighbours."""
    collected = {cid: [] for cid in CAT_IDS}
    for clause in segment(comment):
        v = embed([clause])[0] - CENTRE
        n = np.linalg.norm(v)
        if n == 0:
            continue
        v = v / n
        sims = {cid: round(float(v @ ANCHORS[cid]), 6) for cid in CAT_IDS}
        hits = [cid for cid in CAT_IDS if sims[cid] >= THRESH[cid]]
        if not hits:                     # every clause must land somewhere
            hits = [max(sims, key=sims.get)]
        for cid in hits:
            collected[cid].append(rescale(float(v @ AXES[cid]), SCALES[cid]))
    return {cid: (round(float(np.mean(s)), 4) if s else 0.0)
            for cid, s in collected.items()}


print("Inference pipeline ready.")

# Quick check
demo = "Battery lasts all day but the keyboard feels cheap."
print(f"\nTest: {demo}")
for cid, v in analyse(demo).items():
    if v != 0.0:
        print(f"   {NAMES[cid]:<34}{v:+.3f}")

Inference pipeline ready.

Test: Battery lasts all day but the keyboard feels cheap.
   Battery Life                      +1.000
   Build Quality                     -0.979
   Purchase And Value Perception     -0.709


In [6]:
print("Upload Laptop_Train_v2.csv")
up = files.upload()
df = pd.read_csv(io.BytesIO(list(up.values())[0]))

# 45 rows are denoted as "conflict" - both polarities for one aspect
# Dropping them
n_conflict = int((df["polarity"] == "conflict").sum())
df = df[df["polarity"] != "conflict"].copy()

print(f"\nAspect annotations : {len(df):,}  ({n_conflict} conflict rows dropped)")
print(f"Unique sentences   : {df['Sentence'].nunique():,}")
print(f"Unique aspect terms: {df['Aspect Term'].nunique():,}")

mixed = df.groupby("id")["polarity"].nunique()
MIXED_IDS = set(mixed[mixed > 1].index)
print(f"Mixed-polarity sentences: {len(MIXED_IDS)}")

Upload Laptop_Train_v2.csv


Saving Laptop_Train_v2.csv to Laptop_Train_v2.csv

Aspect annotations : 2,313  (45 conflict rows dropped)
Unique sentences   : 1,456
Unique aspect terms: 1,031
Mixed-polarity sentences: 165


---
# 1 · Determinism audit

Four experiments, each attacking reproducibility from a different angle.
All should report 100%.

**This is the headline result.**

In [7]:
AUDIT_N  = 200    # raise for the final reported figure
N_REPEAT = 5      # raise to 20 for the final reported figure

audit_sentences = df["Sentence"].drop_duplicates().tolist()[:AUDIT_N]


def fingerprint(result):
    """Collapse one comment's result into a comparable string."""
    return "|".join(f"{k}:{v}" for k, v in sorted(result.items()))


print("=" * 70)
print(f"DETERMINISM AUDIT  ·  {len(audit_sentences)} sentences")
print("=" * 70)

t0 = time.time()
baseline = [fingerprint(analyse(s)) for s in audit_sentences]
print(f"\nBaseline pass complete ({time.time()-t0:.0f}s)\n")

audit = {}

# 1 — repeated runs
matches = 0
for _ in range(N_REPEAT):
    current = [fingerprint(analyse(s)) for s in audit_sentences]
    matches += sum(1 for a, b in zip(baseline, current) if a == b)
audit["Repeated runs"] = matches / (len(audit_sentences) * N_REPEAT)
print(f"1. Repeated runs (x{N_REPEAT})            {audit['Repeated runs']:.2%}")

# 2 — shuffled order
order = list(range(len(audit_sentences)))
random.Random(7).shuffle(order)
shuffled = {i: fingerprint(analyse(audit_sentences[i])) for i in order}
audit["Shuffled order"] = sum(
    1 for i in range(len(audit_sentences)) if shuffled[i] == baseline[i]
) / len(audit_sentences)
print(f"2. Dataset order shuffled          {audit['Shuffled order']:.2%}")

# 3 — split and recombine
half = len(audit_sentences) // 2
combined = ([fingerprint(analyse(s)) for s in audit_sentences[:half]] +
            [fingerprint(analyse(s)) for s in audit_sentences[half:]])
audit["Split and recombine"] = sum(
    1 for a, b in zip(baseline, combined) if a == b
) / len(audit_sentences)
print(f"3. Split, processed, recombined    {audit['Split and recombine']:.2%}")

# 4 — complete isolation
isolated = [fingerprint(analyse(s)) for s in audit_sentences]
audit["Isolated"] = sum(
    1 for a, b in zip(baseline, isolated) if a == b
) / len(audit_sentences)
print(f"4. Each sentence in isolation      {audit['Isolated']:.2%}")

print("\n" + "-" * 70)
print("ALL EXPERIMENTS AT 100%. DETERMINISM HOLDS."
      if all(v == 1.0 for v in audit.values())
      else "WARNING: not all at 100%. Check batch_size=1 in embed().")
print("-" * 70)

DETERMINISM AUDIT  ·  200 sentences

Baseline pass complete (20s)

1. Repeated runs (x5)            100.00%
2. Dataset order shuffled          100.00%
3. Split, processed, recombined    100.00%
4. Each sentence in isolation      100.00%

----------------------------------------------------------------------
ALL EXPERIMENTS AT 100%. DETERMINISM HOLDS.
----------------------------------------------------------------------


---
# 2 · Category quality

Four measures, none requiring labels.

In [8]:
print("=" * 70)
print("2.1  ANCHOR SEPARATION")
print("=" * 70)

sep = []
for i in range(len(CAT_IDS)):
    for j in range(i + 1, len(CAT_IDS)):
        a, b = CAT_IDS[i], CAT_IDS[j]
        sep.append((NAMES[a], NAMES[b], float(np.dot(ANCHORS[a], ANCHORS[b]))))
sep.sort(key=lambda x: x[2], reverse=True)

print(f"\n{'Pair':<62}{'Cosine':>9}")
print("-" * 72)
for a, b, s in sep[:8]:
    print(f"{a[:29]:<30}/ {b[:29]:<30}{s:>+9.4f}")

max_sep = sep[0][2] if sep else 0.0
print(f"\nHighest pairwise similarity: {max_sep:+.4f}")
print(f"Range: {min(s for _,_,s in sep):+.4f} to {max_sep:+.4f}")

2.1  ANCHOR SEPARATION

Pair                                                             Cosine
------------------------------------------------------------------------
Hardware Performance          / System Reliability              +0.1989
User Interface Design         / Ease Of Use                     +0.1830
User Interface Design         / Upgrade Options                 +0.1592
Hardware Performance          / Upgrade Options                 +0.1428
Battery Life                  / Hardware Performance            +0.1217
Upgrade Options               / System Reliability              +0.1023
Upgrade Options               / After Sales Support And Warra   +0.1016
Battery Life                  / After Sales Support And Warra   +0.0799

Highest pairwise similarity: +0.1989
Range: -0.2060 to +0.1989


In [9]:
print("=" * 70)
print("2.2  COVERAGE OF GOLD ASPECT TERMS")
print("=" * 70)

terms = df["Aspect Term"].astype(str).unique().tolist()
print(f"\nEmbedding {len(terms):,} unique gold aspect terms…")

TV = embed(terms) - CENTRE
TV = TV / np.linalg.norm(TV, axis=1, keepdims=True)
A  = np.vstack([ANCHORS[c] for c in CAT_IDS])

sims_matrix = TV @ A.T                # (n_terms x n_categories)
best_idx = sims_matrix.argmax(axis=1)
best_sim = sims_matrix.max(axis=1)

# Reused in section 3 to build the gold category sets
TERM_TO_CAT = {t: CAT_IDS[i] for t, i in zip(terms, best_idx)}

COVERAGE_MIN = 0.25
covered = float((best_sim >= COVERAGE_MIN).mean())

print(f"\nCoverage at cosine >= {COVERAGE_MIN}: {covered:.1%}")
print(f"Mean best-match similarity: {best_sim.mean():+.4f}")

print("\nTerms mapped per category:")
counts = pd.Series([NAMES[TERM_TO_CAT[t]] for t in terms]).value_counts()
for name, n in counts.items():
    print(f"   {name:<36}{n:>5}")

print("\n10 least well-covered terms (candidates for a missing category):")
for i in np.argsort(best_sim)[:10]:
    print(f"   {terms[i][:40]:<42}{best_sim[i]:+.4f}  -> {NAMES[CAT_IDS[best_idx[i]]]}")

2.2  COVERAGE OF GOLD ASPECT TERMS

Embedding 1,031 unique gold aspect terms…

Coverage at cosine >= 0.25: 30.7%
Mean best-match similarity: +0.2224

Terms mapped per category:
   User Interface Design                 229
   Ease Of Use                           203
   Hardware Performance                  166
   Device Size And Weight                122
   After Sales Support And Warranty      101
   Battery Life                           60
   Upgrade Options                        57
   Build Quality                          44
   Purchase And Value Perception          27
   System Reliability                     22

10 least well-covered terms (candidates for a missing category):
   DC jack                                   +0.0520  -> Device Size And Weight
   mic jack                                  +0.0562  -> Ease Of Use
   Time Machine                              +0.0666  -> Ease Of Use
   antiviral program                         +0.0723  -> Ease Of Use
   mute             

In [10]:
# ── Recalibrating the coverage threshold ──────────────────────
# COVERAGE_MIN = 0.25 was chosen without measuring anything. The mean
# best-match is 0.2224, so that threshold sits ABOVE the average and
# fails most terms by construction — it reports where a line was drawn,
# not how well the categories cover the corpus.
#
# Calibrate instead against a KNOWN match: each category's own name and
# description should map to its own anchor. That is what "well covered"
# looks like in this space.

self_sims = []
for c in CATS:
    v = embed([f"{c['name']}. {c['description']}"])[0] - CENTRE
    v = v / np.linalg.norm(v)
    self_sims.append(float(v @ ANCHORS[c["id"]]))

self_mean = float(np.mean(self_sims))

print("=" * 70)
print("COVERAGE, RECALIBRATED")
print("=" * 70)
print(f"\nA category's own description scores : {self_mean:+.4f}")
print(f"Mean gold-term best match           : {best_sim.mean():+.4f}")
print(f"Ratio                               : {best_sim.mean()/self_mean:.2f}\n")

for frac in (0.25, 0.33, 0.50):
    t = self_mean * frac
    print(f"   Covered at {frac:.0%} of self-similarity "
          f"({t:+.4f}): {float((best_sim >= t).mean()):.1%}")

# The distribution is more informative than any single cutoff
print(f"\nDistribution of best-match similarity:")
for p in (10, 25, 50, 75, 90):
    print(f"   {p}th percentile  {np.percentile(best_sim, p):+.4f}")

COVERAGE, RECALIBRATED

A category's own description scores : +0.4957
Mean gold-term best match           : +0.2224
Ratio                               : 0.45

   Covered at 25% of self-similarity (+0.1239): 90.9%
   Covered at 33% of self-similarity (+0.1636): 72.0%
   Covered at 50% of self-similarity (+0.2478): 31.6%

Distribution of best-match similarity:
   10th percentile  +0.1257
   25th percentile  +0.1565
   50th percentile  +0.2000
   75th percentile  +0.2645
   90th percentile  +0.3478


In [11]:
print("=" * 70)
print("2.3  ORPHAN RATE AND COHERENCE")
print("=" * 70)

sample = []
for s in df["Sentence"].drop_duplicates().tolist()[:200]:
    sample.extend(segment(s))
sample = sample[:400]

print(f"\nChecking {len(sample)} real clauses…")

orphans, multi = 0, 0
assigned = {cid: [] for cid in CAT_IDS}

for clause in sample:
    v = embed([clause])[0] - CENTRE
    n = np.linalg.norm(v)
    if n == 0:
        continue
    v = v / n
    sims = {cid: round(float(v @ ANCHORS[cid]), 6) for cid in CAT_IDS}
    hits = [cid for cid in CAT_IDS if sims[cid] >= THRESH[cid]]
    if not hits:
        orphans += 1
        hits = [max(sims, key=sims.get)]
    elif len(hits) > 2:
        multi += 1
    for cid in hits:
        assigned[cid].append(v)

orphan_rate = orphans / len(sample)
multi_rate = multi / len(sample)
print(f"\nOrphan rate           {orphan_rate:.1%}")
print(f"Multi-assignment rate {multi_rate:.1%}")

print(f"\n{'Category':<34}{'Within':>9}{'Between':>10}{'Margin':>9}")
print("-" * 64)
for cid in CAT_IDS:
    vs = assigned[cid]
    if len(vs) < 3:
        continue
    V = np.vstack(vs)
    within = float(np.mean(V @ ANCHORS[cid]))
    others = np.vstack([ANCHORS[o] for o in CAT_IDS if o != cid])
    between = float(np.mean(V @ others.T))
    print(f"{NAMES[cid][:33]:<34}{within:>+9.3f}{between:>+10.3f}{within-between:>+9.3f}")

print("\nA positive margin means clauses sit closer to their own category")
print("than to the others, which is what coherence requires.")

2.3  ORPHAN RATE AND COHERENCE

Checking 311 real clauses…

Orphan rate           26.4%
Multi-assignment rate 4.8%

Category                             Within   Between   Margin
----------------------------------------------------------------
Battery Life                         +0.230    +0.010   +0.220
Build Quality                        +0.147    -0.007   +0.155
Hardware Performance                 +0.248    -0.013   +0.261
User Interface Design                +0.212    -0.014   +0.226
Ease Of Use                          +0.157    -0.008   +0.165
Upgrade Options                      +0.216    +0.006   +0.210
System Reliability                   +0.215    +0.007   +0.208
Device Size And Weight               +0.181    -0.015   +0.195
Purchase And Value Perception        +0.221    +0.002   +0.219
After Sales Support And Warranty     +0.270    -0.005   +0.275

A positive margin means clauses sit closer to their own category
than to the others, which is what coherence requires.


---
# 3 · Accuracy — precision, recall and F1

**This section produces the headline figures.**

In [12]:
EVAL_N = 400     # raise for the final reported figure

eval_ids = df["id"].drop_duplicates().tolist()[:EVAL_N]
sentences = df.groupby("id")["Sentence"].first().to_dict()

# Gold category set per sentence, via the term mapping from 2.2
gold_sets, gold_polarity = {}, {}
for sid in eval_ids:
    rows = df[df["id"] == sid]
    cats = set()
    for _, r in rows.iterrows():
        cid = TERM_TO_CAT.get(str(r["Aspect Term"]))
        if cid:
            cats.add(cid)
            gold_polarity[(sid, cid)] = r["polarity"]
    gold_sets[sid] = cats

print(f"Running inference on {len(eval_ids)} sentences…")
t0 = time.time()

pred_sets, pred_scores = {}, {}
for k, sid in enumerate(eval_ids):
    sc = analyse(sentences[sid])
    pred_scores[sid] = sc
    pred_sets[sid] = {cid for cid, v in sc.items() if v != 0.0}
    if (k + 1) % 100 == 0:
        print(f"   {k+1}/{len(eval_ids)}  ({time.time()-t0:.0f}s)")

print(f"Done in {time.time()-t0:.0f}s")

Running inference on 400 sentences…
   100/400  (9s)
   200/400  (19s)
   300/400  (28s)
   400/400  (37s)
Done in 37s


In [13]:
print("=" * 70)
print("3.1  CATEGORY ASSIGNMENT  :  PRECISION, RECALL, F1")
print("=" * 70)
print("\n>>> THESE ARE THE FIGURES FOR THE DISSERTATION <<<\n")

rows_out = []
TP = FP = FN = 0

for cid in CAT_IDS:
    tp = sum(1 for s in eval_ids if cid in gold_sets[s] and cid in pred_sets[s])
    fp = sum(1 for s in eval_ids if cid not in gold_sets[s] and cid in pred_sets[s])
    fn = sum(1 for s in eval_ids if cid in gold_sets[s] and cid not in pred_sets[s])
    TP += tp; FP += fp; FN += fn

    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0

    rows_out.append({"Category": NAMES[cid], "Support": tp + fn,
                     "Precision": round(p, 3), "Recall": round(r, 3),
                     "F1": round(f1, 3)})

micro_p  = TP / (TP + FP) if (TP + FP) else 0.0
micro_r  = TP / (TP + FN) if (TP + FN) else 0.0
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0

results = pd.DataFrame(rows_out).sort_values("Support", ascending=False)
macro_f1 = results["F1"].mean()

print(results.to_string(index=False))
print("\n" + "-" * 64)
print(f"  MICRO-AVERAGED   P {micro_p:.3f}   R {micro_r:.3f}   F1 {micro_f1:.3f}")
print(f"  MACRO-AVERAGED   F1 {macro_f1:.3f}")
print("-" * 64)
print(f"\n  n = {len(eval_ids)} sentences, {len(CAT_IDS)} categories")

3.1  CATEGORY ASSIGNMENT  :  PRECISION, RECALL, F1

>>> THESE ARE THE FIGURES FOR THE DISSERTATION <<<

                        Category  Support  Precision  Recall    F1
                     Ease Of Use      106      0.615   0.302 0.405
            Hardware Performance      103      0.698   0.291 0.411
           User Interface Design       99      0.745   0.354 0.479
          Device Size And Weight       58      0.413   0.328 0.365
After Sales Support And Warranty       57      0.369   0.912 0.525
                    Battery Life       46      0.533   0.870 0.661
   Purchase And Value Perception       27      0.241   0.778 0.368
                   Build Quality       21      0.271   0.905 0.418
                 Upgrade Options       20      0.170   0.400 0.239
              System Reliability        8      0.015   0.125 0.026

----------------------------------------------------------------
  MICRO-AVERAGED   P 0.380   R 0.472   F1 0.421
  MACRO-AVERAGED   F1 0.390
-----------------

In [14]:
print("=" * 70)
print("3.2  MIXED-POLARITY SUBSET")
print("=" * 70)
print("\nSentences carrying opposing sentiment about different aspects.")
print("No single-label system can score above chance here.\n")

mixed_eval = [s for s in eval_ids if s in MIXED_IDS]

if len(mixed_eval) < 5:
    print(f"Only {len(mixed_eval)} mixed sentences in this sample — raise EVAL_N.")
    mixed_f1 = None
else:
    tp = sum(len(gold_sets[s] & pred_sets[s]) for s in mixed_eval)
    fp = sum(len(pred_sets[s] - gold_sets[s]) for s in mixed_eval)
    fn = sum(len(gold_sets[s] - pred_sets[s]) for s in mixed_eval)

    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    mixed_f1 = 2 * p * r / (p + r) if (p + r) else 0.0

    print(f"  n = {len(mixed_eval)} mixed-polarity sentences")
    print(f"  Precision {p:.3f}   Recall {r:.3f}   F1 {mixed_f1:.3f}")
    print(f"\n  Categories per sentence: "
          f"predicted {np.mean([len(pred_sets[s]) for s in mixed_eval]):.2f}, "
          f"gold {np.mean([len(gold_sets[s]) for s in mixed_eval]):.2f}")

3.2  MIXED-POLARITY SUBSET

Sentences carrying opposing sentiment about different aspects.
No single-label system can score above chance here.

  n = 52 mixed-polarity sentences
  Precision 0.430   Recall 0.377   F1 0.402

  Categories per sentence: predicted 1.79, gold 2.04


In [15]:
print("=" * 70)
print("3.3  SENTIMENT ACCURACY")
print("=" * 70)

NEUTRAL_BAND = 0.15    # fixed before evaluation; state this in the write-up


def to_class(score):
    if score >  NEUTRAL_BAND: return "positive"
    if score < -NEUTRAL_BAND: return "negative"
    return "neutral"


GOLD_NUM = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}
CLASSES  = ["positive", "neutral", "negative"]

correct = total = 0
abs_errors = []
confusion = {g: {p: 0 for p in CLASSES} for g in CLASSES}

for sid in eval_ids:
    for cid in gold_sets[sid] & pred_sets[sid]:
        gold = gold_polarity.get((sid, cid))
        if gold is None:
            continue
        pred = to_class(pred_scores[sid][cid])
        confusion[gold][pred] += 1
        total += 1
        correct += (pred == gold)
        abs_errors.append(abs(pred_scores[sid][cid] - GOLD_NUM[gold]))

if total == 0:
    print("\nNo overlapping predictions to score.")
    sent_acc, sent_mae = 0.0, 0.0
else:
    sent_acc = correct / total
    sent_mae = float(np.mean(abs_errors))
    print(f"\n  Three-class accuracy : {sent_acc:.3f}   (n = {total})")
    print(f"  Mean absolute error  : {sent_mae:.3f}")
    print(f"  Neutral band         : +/- {NEUTRAL_BAND}")
    print(f"\n  Confusion matrix (rows = gold, columns = predicted)")
    print(f"  {'':<12}{'positive':>10}{'neutral':>10}{'negative':>10}")
    for g in CLASSES:
        row = confusion[g]
        print(f"  {g:<12}{row['positive']:>10}{row['neutral']:>10}{row['negative']:>10}")

3.3  SENTIMENT ACCURACY

  Three-class accuracy : 0.724   (n = 257)
  Mean absolute error  : 0.485
  Neutral band         : +/- 0.15

  Confusion matrix (rows = gold, columns = predicted)
                positive   neutral  negative
  positive           111         8         3
  neutral             18        12        11
  negative            12        19        63


In [16]:
print("=" * 70)
print("3.4  BASELINE  :  VADER")
print("=" * 70)

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

v_correct = v_total = 0
v_errors = []

for sid in eval_ids:
    v_score = vader.polarity_scores(sentences[sid])["compound"]
    v_class = to_class(v_score)
    # VADER produces ONE score per sentence, compared against every gold
    # aspect in it - which is precisely the limitation being demonstrated.
    for cid in gold_sets[sid]:
        gold = gold_polarity.get((sid, cid))
        if gold is None:
            continue
        v_total += 1
        v_correct += (v_class == gold)
        v_errors.append(abs(v_score - GOLD_NUM[gold]))

vader_acc = v_correct / v_total if v_total else 0.0
vader_mae = float(np.mean(v_errors)) if v_errors else 0.0

print(f"\n{'System':<32}{'3-class acc':>13}{'MAE':>10}")
print("-" * 56)
if total:
    print(f"{'Ours (per-category axes)':<32}{sent_acc:>13.3f}{sent_mae:>10.3f}")
print(f"{'VADER (one global score)':<32}{vader_acc:>13.3f}{vader_mae:>10.3f}")
print("-" * 56)
print("\nVADER assigns one score to the whole sentence, so it cannot")
print("distinguish opposing sentiment about different aspects.")

3.4  BASELINE  :  VADER

System                            3-class acc       MAE
--------------------------------------------------------
Ours (per-category axes)                0.724     0.485
VADER (one global score)                0.560     0.664
--------------------------------------------------------

VADER assigns one score to the whole sentence, so it cannot
distinguish opposing sentiment about different aspects.


---
# 4 · Robustness

Determinism means *identical* input gives identical output. Robustness means
*near-identical* input gives *near-identical* output. Different properties.

Unlike section 1, the target here is **not** 100%. Graceful degradation is the
correct behaviour.

In [17]:
ROBUST_N = 100
robust_sentences = df["Sentence"].drop_duplicates().tolist()[:ROBUST_N]


def lowercase(s):   return s.lower()
def strip_punct(s): return re.sub(r"[^\w\s]", "", s)

def typos(s, rate=0.05):
    """Deterministically swap adjacent characters in some words."""
    rng = random.Random(42)
    out = []
    for w in s.split():
        if len(w) > 3 and rng.random() < rate:
            i = rng.randint(0, len(w) - 2)
            w = w[:i] + w[i+1] + w[i] + w[i+2:]
        out.append(w)
    return " ".join(out)

def truncate(s):
    words = s.split()
    return " ".join(words[:max(3, int(len(words) * 0.8))])


PERTURBATIONS = {
    "Lowercased":          lowercase,
    "Punctuation removed": strip_punct,
    "Character typos":     typos,
    "Truncated to 80%":    truncate,
}

print("=" * 70)
print("ROBUSTNESS UNDER PERTURBATION")
print("=" * 70)
print(f"\nBaseline on {len(robust_sentences)} sentences…")

base_sets, base_scores = {}, {}
for s in robust_sentences:
    sc = analyse(s)
    base_scores[s] = sc
    base_sets[s] = {c for c, v in sc.items() if v != 0.0}

print(f"\n{'Perturbation':<24}{'Jaccard':>10}{'Exact set':>12}{'Sentiment MAE':>16}")
print("-" * 64)

robustness = {}
for label, fn in PERTURBATIONS.items():
    jac, exact, maes = [], 0, []
    for s in robust_sentences:
        sc = analyse(fn(s))
        st = {c for c, v in sc.items() if v != 0.0}
        union = base_sets[s] | st
        jac.append(len(base_sets[s] & st) / len(union) if union else 1.0)
        exact += (st == base_sets[s])
        for cid in base_sets[s] & st:
            maes.append(abs(sc[cid] - base_scores[s][cid]))
    robustness[label] = float(np.mean(jac))
    print(f"{label:<24}{np.mean(jac):>10.3f}{exact/len(robust_sentences):>12.1%}"
          f"{(np.mean(maes) if maes else 0):>16.3f}")

print("-" * 64)
print("\nJaccard measures overlap of category sets against the unperturbed")
print("original. High values mean the system is stable under noisy input.")

ROBUSTNESS UNDER PERTURBATION

Baseline on 100 sentences…

Perturbation               Jaccard   Exact set   Sentiment MAE
----------------------------------------------------------------
Lowercased                   0.992       98.0%           0.008
Punctuation removed          0.836       72.0%           0.067
Character typos              0.711       53.0%           0.119
Truncated to 80%             0.717       51.0%           0.122
----------------------------------------------------------------

Jaccard measures overlap of category sets against the unperturbed
original. High values mean the system is stable under noisy input.


---
# 5 · Summary

In [18]:
print("=" * 70)
print("RESULTS SUMMARY")
print(f"Model: {artifact['name']}  ({artifact['model_id']})")
print("=" * 70)

print("\n1 · REPRODUCIBILITY")
for k, v in audit.items():
    print(f"     {k:<32}{v:>10.2%}")

print("\n2 · CATEGORY QUALITY")
print(f"     {'Highest anchor similarity':<32}{max_sep:>+10.4f}")
print(f"     {'Gold term coverage':<32}{covered:>10.1%}")
print(f"     {'Orphan rate':<32}{orphan_rate:>10.1%}")
print(f"     {'Multi-assignment rate':<32}{multi_rate:>10.1%}")

print("\n3 · ACCURACY")
print(f"     {'Micro precision':<32}{micro_p:>10.3f}")
print(f"     {'Micro recall':<32}{micro_r:>10.3f}")
print(f"     {'Micro F1':<32}{micro_f1:>10.3f}")
print(f"     {'Macro F1':<32}{macro_f1:>10.3f}")
if mixed_f1 is not None:
    print(f"     {'Mixed-polarity subset F1':<32}{mixed_f1:>10.3f}")
if total:
    print(f"     {'Sentiment 3-class accuracy':<32}{sent_acc:>10.3f}")
    print(f"     {'Sentiment MAE':<32}{sent_mae:>10.3f}")
    print(f"     {'VADER 3-class accuracy':<32}{vader_acc:>10.3f}")

print("\n4 · ROBUSTNESS (Jaccard against unperturbed)")
for k, v in robustness.items():
    print(f"     {k:<32}{v:>10.3f}")

print("\n" + "=" * 70)
print("PER-CATEGORY BREAKDOWN")
print("=" * 70)
print(results.to_string(index=False))

results.to_csv("evaluation_results.csv", index=False)
files.download("evaluation_results.csv")
print("\nSaved evaluation_results.csv")

RESULTS SUMMARY
Model: Laptop Review Analysis  (mdl_260825014042)

1 · REPRODUCIBILITY
     Repeated runs                      100.00%
     Shuffled order                     100.00%
     Split and recombine                100.00%
     Isolated                           100.00%

2 · CATEGORY QUALITY
     Highest anchor similarity          +0.1989
     Gold term coverage                   30.7%
     Orphan rate                          26.4%
     Multi-assignment rate                 4.8%

3 · ACCURACY
     Micro precision                      0.380
     Micro recall                         0.472
     Micro F1                             0.421
     Macro F1                             0.390
     Mixed-polarity subset F1             0.402
     Sentiment 3-class accuracy           0.724
     Sentiment MAE                        0.485
     VADER 3-class accuracy               0.560

4 · ROBUSTNESS (Jaccard against unperturbed)
     Lowercased                           0.992
     Punctuatio

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Saved evaluation_results.csv


---
# 6 · Optional — the generative baseline

**This produces the most important single number in the dissertation.**

100% reproducibility means little on its own. It is meaningful only against
something that is *not* reproducible — which is what a practitioner would
actually use today.

This section asks the same language model to categorise and score comments
directly, at temperature zero, repeatedly, and measures how often it agrees
with itself.

**Requirements:** a GPU runtime, and about fifteen minutes. Skip if short on
time — sections 1–5 stand on their own, but H1 explicitly predicts this
contrast.

In [19]:
# Install and start Ollama
!apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull qwen2.5:14b
print("\nReady.")

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Ready.


In [20]:
import requests

OLLAMA_URL = "http://localhost:11434"
LLM_MODEL  = "qwen2.5:14b"

CATEGORY_LIST = "\n".join(f"- {NAMES[c]}" for c in CAT_IDS)

BASELINE_SYSTEM = f"""You categorise customer feedback and score sentiment.

Assign each comment to one or more of these categories:
{CATEGORY_LIST}

For each category the comment addresses, give a sentiment score between
-1.0 (very negative) and +1.0 (very positive).

Only include categories the comment actually discusses.

Reply with JSON only.
Format: {{"categories": {{"Category Name": 0.5}}}}"""


def llm_categorise(comment, retries=2):
    """
    Ask the model to categorise directly — the approach this dissertation
    argues against. Temperature zero and a fixed seed, exactly as a
    practitioner attempting reproducibility would configure it.
    """
    for _ in range(retries + 1):
        try:
            r = requests.post(
                f"{OLLAMA_URL}/api/chat",
                json={
                    "model": LLM_MODEL,
                    "messages": [
                        {"role": "system", "content": BASELINE_SYSTEM},
                        {"role": "user", "content": comment},
                    ],
                    "stream": False,
                    "format": "json",
                    "options": {"temperature": 0, "seed": 42, "num_ctx": 4096},
                },
                timeout=300,
            )
            r.raise_for_status()
            out = json.loads(r.json()["message"]["content"])
            cats = out.get("categories", {})
            if isinstance(cats, dict):
                # Round to the same precision our system reports
                return {k: round(float(v), 4)
                        for k, v in cats.items()
                        if isinstance(v, (int, float))}
        except Exception:
            continue
    return {}


# Smoke test
print("Test:", llm_categorise("Battery lasts all day but the keyboard feels cheap."))

Test: {'Battery Life': 1.0, 'Build Quality': -0.5}


In [ ]:
BASELINE_N       = 100   # comments
BASELINE_REPEATS = 5     # runs; raise to 20 for the reported figure

baseline_sentences = df["Sentence"].drop_duplicates().tolist()[:BASELINE_N]

print("=" * 70)
print(f"GENERATIVE BASELINE  ·  {BASELINE_N} comments x {BASELINE_REPEATS} runs")
print("=" * 70)
print("\nThis is slow — roughly a second per comment per run.\n")


def bl_fingerprint(d):
    """Same comparison used for our own system."""
    return "|".join(f"{k}:{v}" for k, v in sorted(d.items()))


runs = []
for rep in range(BASELINE_REPEATS):
    t0 = time.time()
    runs.append([bl_fingerprint(llm_categorise(s)) for s in baseline_sentences])
    print(f"   run {rep+1}/{BASELINE_REPEATS} complete ({time.time()-t0:.0f}s)")

# How often does every run agree for a given comment?
stable = sum(1 for i in range(BASELINE_N)
             if len({runs[r][i] for r in range(BASELINE_REPEATS)}) == 1)

baseline_stability = stable / BASELINE_N
baseline_disagreement = 1 - baseline_stability

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)
print(f"\n  Generative baseline, identical across all {BASELINE_REPEATS} runs:"
      f"  {baseline_stability:.1%}")
print(f"  Disagreement rate:                             "
      f"  {baseline_disagreement:.1%}")
print(f"\n  Our system, identical across all runs:         "
      f"  {audit['Repeated runs']:.1%}")

print("\n" + "-" * 70)
print("This contrast is the central result of the dissertation.")
print("-" * 70)

# Show examples of the baseline disagreeing with itself
print("\nExamples where the baseline changed its answer:\n")
shown = 0
for i in range(BASELINE_N):
    answers = {runs[r][i] for r in range(BASELINE_REPEATS)}
    if len(answers) > 1 and shown < 3:
        print(f'  "{baseline_sentences[i][:78]}"')
        for a in list(answers)[:3]:
            print(f"     -> {a[:76]}")
        print()
        shown += 1

GENERATIVE BASELINE  ·  100 comments x 5 runs

This is slow — roughly a second per comment per run.

   run 1/5 complete (106s)
   run 2/5 complete (130s)


---

## Recording the results

Replace the placeholders in your abstract and the "target" values on the
evaluation slide with the measured figures above.

**Report honestly.** If a category scores badly, show it. An examiner will find
it in the appendix anyway, and volunteering it reads far better than being
caught. A weak category is also a finding — it usually means the anchor is
poorly separated, or the gold terms mapping onto it are heterogeneous.

**Caveats to state alongside the numbers:**

- Gold aspect terms are mapped to discovered categories by cosine similarity,
  so the mapping contributes its own error. These figures are a **lower bound**.
- Thresholds were calibrated on machine-generated exemplars, which may be
  cleaner than real customer text.
- 45 conflict-annotated rows were excluded, following standard practice.
- The neutral band was fixed at ±0.15 before evaluation, not tuned afterwards.
- SemEval annotates for a linguistic purpose rather than an operational one;
  some apparent errors are annotation-purpose mismatches rather than system
  failures. An error analysis distinguishing the two would strengthen the
  discussion.